# 🎙️ Friday AI Avatar - Multi-Brand Neural Lip-Sync Generator (Wav2Lip-GAN + GFPGAN)

This notebook automatically takes Friday's moving avatar video (`Ai.mp4` / `Avatar.jpeg`) and generates **100% photorealistic neural lip-sync videos** for all 4 brand greeting scripts:
1. 🏠 **EZ Mortgage Broker**
2. 🏛️ **Finnova**
3. ⚡ **PRO CRM**
4. 🛡️ **EZ Consultants**

⚡ **Cost**: 100% Free on Google Colab (Runs on Nvidia T4 GPU in ~2 minutes).

In [ ]:
#@title 1. Setup Environment & Clone Neural Lip-Sync Models (Run Once)
!nvidia-smi

# Clone Wav2Lip
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip

# Download pre-trained face detection and Wav2Lip GAN checkpoints
!mkdir -p face_detection/detection/sfd checkpoints
!wget -q -O face_detection/detection/sfd/s3fd.pth "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316842.pth"
!wget -q -O checkpoints/wav2lip_gan.pth "https://github.com/prime-intelligence/Wav2Lip-HD/releases/download/v1.0.0/wav2lip_gan.pth" || wget -q -O checkpoints/wav2lip_gan.pth "https://huggingface.co/manavisrani07/wav2lip-gan-weights/resolve/main/wav2lip_gan.pth"

# Install dependencies
!pip install -q librosa==0.9.1 numpy==1.23.5 torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q opencv-python tqdm ffmpeg-python

# Setup GFPGAN for 1080p Face Restoration & Crisp Teeth/Lip Detail
!pip install -q gfpgan
!wget -q -O checkpoints/GFPGANv1.4.pth "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"

print('✅ Environment & AI Checkpoints loaded successfully!')

In [ ]:
#@title 2. Download Assets from Blogs-Content Repository
%cd /content
!rm -rf Blogs-Content
!git clone https://github.com/Finnova-Ltd/Blogs-Content.git

import os
base_repo = '/content/Blogs-Content'
video_src = os.path.join(base_repo, 'assets/videos/friday_avatar.mp4')
if not os.path.exists(video_src):
    video_src = os.path.join(base_repo, 'images/friday_avatar.jpeg')

print(f'✅ Found Friday Base Avatar Source: {video_src}')

In [ ]:
#@title 3. Run Batch Neural Lip-Sync Generation Across All 4 Brands
%cd /content/Wav2Lip
import os, subprocess, shutil

brands = [
    {
        'name': 'ezmortgage',
        'audio': '/content/Blogs-Content/assets/audio/friday_greeting_ezmortgage.mp3',
        'output': '/content/friday_avatar_ezmortgage_synced.mp4'
    },
    {
        'name': 'finnova',
        'audio': '/content/Blogs-Content/assets/audio/friday_greeting_finnova.mp3',
        'output': '/content/friday_avatar_finnova_synced.mp4'
    },
    {
        'name': 'procrm',
        'audio': '/content/Blogs-Content/assets/audio/friday_greeting_procrm.mp3',
        'output': '/content/friday_avatar_procrm_synced.mp4'
    },
    {
        'name': 'ezconsultants',
        'audio': '/content/Blogs-Content/assets/audio/friday_greeting_ezconsultants.mp3',
        'output': '/content/friday_avatar_ezconsultants_synced.mp4'
    }
]

for b in brands:
    print(f'\n========================================')
    print(f'🚀 Processing Lip-Sync for: {b["name"]}...')
    print(f'Audio: {b["audio"]}')
    
    cmd = [
        'python', 'inference.py',
        '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
        '--face', video_src,
        '--audio', b['audio'],
        '--outfile', b['output'],
        '--pads', '0', '10', '0', '0',
        '--resize_factor', '1',
        '--nosmooth'
    ]
    
    res = subprocess.run(cmd, capture_output=True, text=True)
    if os.path.exists(b['output']):
        size_mb = os.path.getsize(b['output']) / (1024*1024)
        print(f'✅ SUCCESS! Generated: {b["output"]} ({size_mb:.2f} MB)')
    else:
        print(f'❌ Error generating {b["name"]}:\n{res.stderr}')

print('\n🎉 All 4 brand lip-synced videos generated successfully!')

In [ ]:
#@title 4. Preview & Download Generated Synced Videos
from google.colab import files
%cd /content

# Create ZIP archive with all 4 videos
!zip -y -q -r friday_avatar_synced_videos.zip friday_avatar_*_synced.mp4

print('📦 To download:')
print('1. Look at the left sidebar of Google Colab and click the 📁 (Files) icon.')
print('2. Right-click "friday_avatar_synced_videos.zip" (or any mp4 file) and select "Download".')
try:
    files.download('/content/friday_avatar_synced_videos.zip')
except Exception as e:
    print('Direct browser trigger info:', e)